# stareControl — 視線・瞬き検出（Colab パイロット）
通常のWebカメラで約12秒の動画を撮影し、MediaPipe **Tasks API** で虹彩の相対位置とウインクを解析します。

**注意**: Google ColabからPC本体のマウスは操作しません。画面座標へのキャリブレーションも未実装です。動画はColabランタイム上に保存されます。カメラ映像には個人情報が含まれるため、共有する際はご注意ください。

**手順**: 上からセルを順に実行 → カメラの使用を許可 → 視線を上下左右に動かす → 左右のウインクを試す → 注釈付き動画とグラフを確認します。

In [ ]:
%pip -q install -U mediapipe opencv-python-headless pandas matplotlib

## 1. カメラで12秒間撮影
Chrome等のカメラが利用できるブラウザで開いてください。カメラ使用を許可すると録画が始まります。

In [ ]:
from google.colab.output import eval_js
from IPython.display import Javascript, display
import base64, os, subprocess

RECORD_SECONDS = 12
js = r"""
async function recordCamera(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({video:{width:640,height:480,frameRate:24},audio:false});
  const wrap=document.createElement('div');
  const video=document.createElement('video');
  const info=document.createElement('p');
  video.srcObject=stream;video.autoplay=true;video.muted=true;video.playsInline=true;
  video.style='width:min(100%,640px);border-radius:12px;transform:scaleX(-1)';
  wrap.append(video,info);document.body.append(wrap);
  try{
    await video.play();
    const mime=['video/webm;codecs=vp8','video/webm','video/mp4'].find(x=>MediaRecorder.isTypeSupported(x));
    if(!mime)throw new Error('MediaRecorder is unsupported. Try Chrome.');
    const chunks=[];const recorder=new MediaRecorder(stream,{mimeType:mime});
    recorder.ondataavailable=e=>{if(e.data.size)chunks.push(e.data)};
    const done=new Promise((resolve,reject)=>{recorder.onstop=resolve;recorder.onerror=e=>reject(e.error||e)});
    recorder.start(1000);
    const start=Date.now();
    const timer=setInterval(()=>{info.textContent='Recording: '+Math.max(0,Math.ceil(seconds-(Date.now()-start)/1000))+' seconds left';},200);
    await new Promise(r=>setTimeout(r,seconds*1000));
    clearInterval(timer);recorder.stop();await done;
    const blob=new Blob(chunks,{type:mime});
    const data=await new Promise((resolve,reject)=>{const fr=new FileReader();fr.onload=()=>resolve(fr.result);fr.onerror=reject;fr.readAsDataURL(blob)});
    return {mime,data:data.split(',')[1]};
  }finally{stream.getTracks().forEach(t=>t.stop());video.srcObject=null;wrap.remove()}
}
"""
display(Javascript(js))
recorded=eval_js(f"recordCamera({RECORD_SECONDS})")
src='/content/gaze_recording.'+('mp4' if 'mp4' in recorded['mime'] else 'webm')
with open(src,'wb') as f: f.write(base64.b64decode(recorded['data']))
VIDEO_PATH='/content/gaze_recording.mp4'
subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y','-i',src,'-an','-vf','fps=15','-c:v','mpeg4','-q:v','4',VIDEO_PATH],check=True)
print('Saved:',VIDEO_PATH,round(os.path.getsize(VIDEO_PATH)/1e6,2),'MB')

## 2. Face Landmarkerで虹彩・瞬きを解析
MediaPipeの新しいTasks APIを使用します。旧式の `mp.solutions` は不要です。しきい値は個人差があり、左右クリックの割り当ては鏡像表示で確認・調整してください。

In [ ]:
import cv2, mediapipe as mp, numpy as np, pandas as pd
from pathlib import Path
from IPython.display import Video, display
import urllib.request

BLINK_EAR=0.19
WINK_MIN_FRAMES=3
WINK_MAX_FRAMES=18
CLICK_COOLDOWN_FRAMES=8

EYES={
 'eye_a': {'corners':(33,133),'ear':(33,160,158,133,153,144),'iris':468,'lids':(159,145)},
 'eye_b': {'corners':(362,263),'ear':(362,385,387,263,373,380),'iris':473,'lids':(386,374)}
}
def dist(p,q): return float(np.linalg.norm(np.asarray(p)-np.asarray(q)))
def measure(lm,s):
    point=lambda i:(lm[i].x,lm[i].y)
    ids=s['ear']
    ear=(dist(point(ids[1]),point(ids[5]))+dist(point(ids[2]),point(ids[4])))/(2*max(dist(point(ids[0]),point(ids[3])),1e-8))
    xs=sorted((point(s['corners'][0])[0],point(s['corners'][1])[0]))
    ys=sorted((point(s['lids'][0])[1],point(s['lids'][1])[1]))
    ix,iy=point(s['iris'])
    return {'ear':ear,'gx':(ix-xs[0])/max(xs[1]-xs[0],1e-8),
            'gy':(iy-ys[0])/max(ys[1]-ys[0],1e-8),'ix':ix,'iy':iy}

MODEL_PATH='/content/face_landmarker.task'
MODEL_URL='https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
if not Path(MODEL_PATH).exists(): urllib.request.urlretrieve(MODEL_URL,MODEL_PATH)

cap=cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened(): raise RuntimeError('Could not open recording. Run the camera cell again.')
fps=cap.get(cv2.CAP_PROP_FPS) or 15
w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
OUTPUT_VIDEO='/content/gaze_annotated.mp4'
writer=cv2.VideoWriter(OUTPUT_VIDEO,cv2.VideoWriter_fourcc(*'mp4v'),fps,(w,h))
if not writer.isOpened(): raise RuntimeError('Could not create output video.')
opts=mp.tasks.vision.FaceLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_faces=1,min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,min_tracking_confidence=0.5)
rows=[]; active={'eye_a':0,'eye_b':0}; last_click=-999
try:
    with mp.tasks.vision.FaceLandmarker.create_from_options(opts) as detector:
        index=0
        while True:
            ok,frame=cap.read()
            if not ok: break
            frame=cv2.flip(frame,1)
            rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
            mpimg=mp.Image(image_format=mp.ImageFormat.SRGB,data=np.ascontiguousarray(rgb))
            result=detector.detect_for_video(mpimg,int(round(index*1000/fps)))
            row={'time_s':round(index/fps,3),'face_detected':False,
                 'gaze_x':np.nan,'gaze_y':np.nan,'eye_a_ear':np.nan,'eye_b_ear':np.nan,'event':''}
            if result.face_landmarks and len(result.face_landmarks[0])>473:
                lm=result.face_landmarks[0]
                m={name:measure(lm,spec) for name,spec in EYES.items()}
                opened=[v for v in m.values() if v['ear']>=BLINK_EAR]
                row.update({'face_detected':True,'eye_a_ear':m['eye_a']['ear'],'eye_b_ear':m['eye_b']['ear']})
                if opened:
                    row['gaze_x']=float(np.mean([v['gx'] for v in opened]))
                    row['gaze_y']=float(np.mean([v['gy'] for v in opened]))
                closed={name:(v['ear']<BLINK_EAR) for name,v in m.items()}
                for name in active:
                    other='eye_b' if name=='eye_a' else 'eye_a'
                    if closed[name] and not closed[other]:
                        active[name]+=1
                    elif active[name]:
                        if WINK_MIN_FRAMES<=active[name]<=WINK_MAX_FRAMES and index-last_click>CLICK_COOLDOWN_FRAMES:
                            row['event']='LEFT_CLICK' if name=='eye_a' else 'RIGHT_CLICK'
                            last_click=index
                        active[name]=0
                for v in m.values():
                    cv2.circle(frame,(int(v['ix']*w),int(v['iy']*h)),4,(0,220,130) if v['ear']>=BLINK_EAR else (0,80,240),-1)
                if opened:
                    cv2.putText(frame,'Relative gaze X:{:.2f} Y:{:.2f}'.format(row['gaze_x'],row['gaze_y']),
                                (12,30),cv2.FONT_HERSHEY_SIMPLEX,.65,(250,250,250),2)
                cv2.putText(frame,'EAR A:{:.2f} B:{:.2f}'.format(m['eye_a']['ear'],m['eye_b']['ear']),
                            (12,57),cv2.FONT_HERSHEY_SIMPLEX,.6,(250,250,250),2)
            else:
                active={'eye_a':0,'eye_b':0}
                cv2.putText(frame,'FACE NOT FOUND',(12,30),cv2.FONT_HERSHEY_SIMPLEX,.65,(30,50,255),2)
            if row['event']:
                cv2.putText(frame,row['event'],(12,95),cv2.FONT_HERSHEY_SIMPLEX,.9,(0,235,255),3)
            writer.write(frame);rows.append(row);index+=1
finally:
    cap.release();writer.release()
df=pd.DataFrame(rows)
CSV_PATH='/content/gaze_results.csv'
df.to_csv(CSV_PATH,index=False)
print('MediaPipe:',mp.__version__,'Frames:',len(df))
print('Face detection rate:',f'{df.face_detected.mean():.1%}' if len(df) else 'no frames')
events=df.loc[df.event!='',['time_s','event']]
print('Click candidates:\n',events.to_string(index=False) if len(events) else 'none')
display(Video(OUTPUT_VIDEO,embed=True,width=700))

## 3. 視線・目の開閉・クリック候補をグラフ表示

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(2,1,figsize=(12,7),sharex=True)
axes[0].plot(df.time_s,df.gaze_x,label='Relative gaze X')
axes[0].plot(df.time_s,df.gaze_y,label='Relative gaze Y')
axes[0].set_ylabel('Uncalibrated relative position')
axes[0].legend();axes[0].grid(alpha=.3)
axes[1].plot(df.time_s,df.eye_a_ear,label='Eye A EAR')
axes[1].plot(df.time_s,df.eye_b_ear,label='Eye B EAR')
axes[1].axhline(BLINK_EAR,linestyle='--',label='Wink threshold')
for _,e in df[df.event!=''].iterrows(): axes[1].axvline(e.time_s,alpha=.5)
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Eye aspect ratio (EAR)')
axes[1].legend();axes[1].grid(alpha=.3)
plt.tight_layout();plt.show()

## 4. CSV・注釈付き動画の保存

In [ ]:
from google.colab import files
files.download(CSV_PATH)
# 動画も必要なら次の行のコメントを外してください。
# files.download(OUTPUT_VIDEO)

## 今後の開発
- 画面上9点のキャリブレーションによる視線座標の推定
- 個人別の瞬き閾値校正と誤クリック抑制
- PySide6を使うWindows/macOSデスクトップアプリへの移植

**制限**: 視線の数値は虹彩の相対位置で、画面上の正確な座標ではありません。左右のイベント名は初期の仮割り当てで、実際に動作を見て確定してください。